# Fase 3 · M05: Target y Exportación

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M05 — Target y Export |

---

## 🎯 Qué hace

Define el target `abandono`, aplica target encoding a `titulacion`, elimina leakage
y exporta D y D_strict como datasets definitivos para modelado.

## 📋 Requisitos

- `data/03_features/df_exp_automl_target.parquet` — generado por M04a
- `data/03_features/df_exp_target_eda.parquet` — generado por M04b

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/dataset_final_tfm.parquet` | D_strict — dataset de producción |
| `data/automl/df_exp_automl_D.parquet` | D — dataset completo sin leakage |
| `data/03_features/df_eda_con_target.parquet` | EDA con target y texto legible |
| `docs/html/fase3/m05_target_export.html` | Informe HTML |

## 🔄 Flujo

```
df_exp_automl_target.parquet (M04a)
    ↓ Calcular target abandono
    ↓ Target encoding: titulacion → tasa_abandono_titulacion
    ↓ Eliminar leakage
    ↓ Exportar D (todas las features) y D_strict (features auditadas)
    → dataset_final_tfm.parquet + df_exp_automl_D.parquet + HTML
```

## ➡️ Siguiente

`f3_m06_alerta_temprana.ipynb` — análisis de alerta temprana


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

from src.config import RUTA_FEATURES, RUTA_AUTOML, RUTA_HTML, info_entorno, DICCIONARIO_RAMAS
from src.utils import cargar_parquet, crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_pagina_desde_fichero

RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_AUTOML, RUTA_FASE3_HTML])

CURSO_REFERENCIA = 2021
ANOS_INACTIVO_UMBRAL = 2
fmt = formato_numero_es

print('✓ Configuración completada')
info_entorno()

✓ Directorios verificados: 3
✓ Configuración completada
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_pro

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASETS
# ============================================================================

print('\n' + '='*70)
print('F3-M05: TARGET (ABANDONO) Y EXPORT FINAL')
print('='*70)

print('\n📥 Cargando datasets M04a (AutoML) y M04b (EDA)...')

df_automl = cargar_parquet(RUTA_FEATURES / 'df_exp_automl_target.parquet')
df_eda = cargar_parquet(RUTA_FEATURES / 'df_exp_target_eda.parquet')

print(f'✓ AutoML: {len(df_automl):,} × {len(df_automl.columns)} columnas')
print(f'✓ EDA: {len(df_eda):,} × {len(df_eda.columns)} columnas')


F3-M05: TARGET (ABANDONO) Y EXPORT FINAL

📥 Cargando datasets M04a (AutoML) y M04b (EDA)...
✓ Cargado: df_exp_automl_target.parquet (33,621 filas, 49 columnas)
✓ Cargado: df_exp_target_eda.parquet (33,621 filas, 49 columnas)
✓ AutoML: 33,621 × 49 columnas
✓ EDA: 33,621 × 49 columnas


In [3]:
# ============================================================================
# CELDA 3: CALCULAR TARGET (ABANDONO)
# ============================================================================

print('\n' + '-'*70)
print('CALCULANDO TARGET (ABANDONO)')
print('-'*70)

print(f'\nFórmula: abandono = (egresado="N") AND (egresado_de_hecho=0) AND ({CURSO_REFERENCIA} - curso_ultimo >= {ANOS_INACTIVO_UMBRAL})')

for df_name, df in [('AutoML', df_automl), ('EDA', df_eda)]:
    df['anos_inactivo'] = CURSO_REFERENCIA - df['curso_ultimo']
    egresado_val = df['egresado'].iloc[0]
    if isinstance(egresado_val, str):
        no_egresado = (df['egresado'] == 'N')
    else:
        no_egresado = (df['egresado'] == 0)
    no_egresado_de_hecho = True
    if 'egresado_de_hecho' in df.columns:
        no_egresado_de_hecho = (df['egresado_de_hecho'] == 0)
    abandono_condition = no_egresado & no_egresado_de_hecho & (df['anos_inactivo'] >= ANOS_INACTIVO_UMBRAL)
    df['abandono'] = abandono_condition.astype(int)

    n_total = len(df)
    n_abandono = (df['abandono'] == 1).sum()
    if isinstance(egresado_val, str):
        n_egresados = (df['egresado'] == 'S').sum()
    else:
        n_egresados = (df['egresado'] == 1).sum()
    n_de_hecho = (df['egresado_de_hecho'] == 1).sum() if 'egresado_de_hecho' in df.columns else 0
    n_activos = n_total - n_egresados - n_de_hecho - n_abandono

    print(f'\n✓ {df_name} ({fmt(n_total)} expedientes):')
    print(f'   🎓 Egresados (título):      {fmt(n_egresados)} ({n_egresados/n_total*100:.1f}%)')
    print(f'   📋 Completaron sin título:   {fmt(n_de_hecho)} ({n_de_hecho/n_total*100:.1f}%)')
    print(f'   ❌ Abandono (target=1):      {fmt(n_abandono)} ({n_abandono/n_total*100:.1f}%)')
    print(f'   ⏳ Activos (< 2 años):       {fmt(n_activos)} ({n_activos/n_total*100:.1f}%)')


----------------------------------------------------------------------
CALCULANDO TARGET (ABANDONO)
----------------------------------------------------------------------

Fórmula: abandono = (egresado="N") AND (egresado_de_hecho=0) AND (2021 - curso_ultimo >= 2)

✓ AutoML (33.621 expedientes):
   🎓 Egresados (título):      12.392 (36.9%)
   📋 Completaron sin título:   170 (0.5%)
   ❌ Abandono (target=1):      9.833 (29.2%)
   ⏳ Activos (< 2 años):       11.226 (33.4%)

✓ EDA (33.621 expedientes):
   🎓 Egresados (título):      12.392 (36.9%)
   📋 Completaron sin título:   170 (0.5%)
   ❌ Abandono (target=1):      9.833 (29.2%)
   ⏳ Activos (< 2 años):       11.226 (33.4%)


In [4]:
# ============================================================================
# CELDA 3b: TARGET ENCODING — titulacion → tasa_abandono_titulacion
# ============================================================================
# Target encoding: sustituir el nombre de la titulación por la tasa histórica
# de abandono de esa titulación. Es la técnica más adecuada aquí porque:
#   - El número tiene significado real y causal (prob. histórica de abandono)
#   - No introduce sesgo de orden (como haría ordinal encoding)
#   - No explota columnas (como haría one-hot con 40 dummies)
#   - Es interpretable ante el tribunal
#
# PLANES NUEVOS (Plan 2017, 2018, 2020):
#   Tienen tasa≈0 porque sus alumnos aún no han tenido tiempo de graduarse
#   (el dataset llega hasta 2021). No es tasa real — son planes en curso.
#   Solución: usar la tasa del plan equivalente anterior (mismo grado, plan antiguo).
#   Si no existe equivalente, usar la media de su rama como fallback.
#
# IMPORTANTE: se calcula sobre TODO el dataset (no solo train) porque es
# información histórica previa — no usamos la respuesta del alumno para
# predecir su propia respuesta. En producción se usa la tasa histórica
# calculada aquí, no del alumno nuevo.
# ============================================================================

print('\n' + '-'*70)
print('TARGET ENCODING: titulacion → tasa_abandono_titulacion')
print('-'*70)

# Tasas calculadas sobre planes CONSOLIDADOS (no nuevos)
# Fuente: df_automl con target abandono ya definido en celda anterior
# Solo usamos titulaciones sin 'Plan' en el nombre (planes consolidados)
mask_consolidado = ~df_automl['titulacion'].astype(str).str.contains('Plan |Doble', na=False)
tasas_base = (
    df_automl[mask_consolidado]
    .groupby('titulacion')['abandono']
    .mean()
    .round(4)
)

# Tasas por rama para fallback
tasas_rama = df_automl.groupby('rama')['abandono'].mean().round(4)
media_global = df_automl['abandono'].mean().round(4)

# Mapa de equivalencias para planes nuevos
# Cada plan nuevo → tasa del plan equivalente consolidado
EQUIVALENCIAS_PLANES = {
    'Grado en Maestro en Educación Primaria (Plan 2018)':       'Grado en Maestro en Educación Primaria',
    'Grado en Maestro en Educación Infantil (Plan 2018)':       'Grado en Maestro en Educación Infantil',
    'Grado en Medicina (Plan 2017)':                            'Grado en Medicina',
    'Grado en Historia y Patrimonio (Plan 2015)':               'Grado en Historia y Patrimonio',
    'Grado en Arquitectura Técnica (Plan 2020)':                'Grado en Arquitectura Técnica',
    'Grado en Criminologia y Seguridad  (Plan 2020)':           'Grado en Criminologia y Seguridad',
    'Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)': 'Grado en Ingeniería Agroalimentaria y del Medio Rural',
    # Doble grado sin equivalente → media de rama SO
    'Doble Grado en Administración y Dirección de Empresas y Derecho,': None,
}

def obtener_tasa(titulacion, rama):
    """Devuelve la tasa de abandono histórica para una titulación.
    Prioridad: tasa propia → equivalente → media rama → media global.
    """
    # 1. Plan consolidado — tasa propia
    if titulacion in tasas_base.index:
        return tasas_base[titulacion]
    # 2. Plan nuevo con equivalente conocido
    if titulacion in EQUIVALENCIAS_PLANES:
        equiv = EQUIVALENCIAS_PLANES[titulacion]
        if equiv and equiv in tasas_base.index:
            return tasas_base[equiv]
    # 3. Fallback: media de rama
    if rama in tasas_rama.index:
        return tasas_rama[rama]
    # 4. Fallback final: media global
    return media_global

# Aplicar a df_automl
df_automl['tasa_abandono_titulacion'] = df_automl.apply(
    lambda row: obtener_tasa(str(row['titulacion']), row['rama']), axis=1
)

# Aplicar a df_eda (titulacion sigue como string, añadimos la tasa numérica)
df_eda['tasa_abandono_titulacion'] = df_eda.apply(
    lambda row: obtener_tasa(str(row['titulacion']), row['rama']), axis=1
)

# Resumen
print(f'\n✓ tasa_abandono_titulacion calculada')
print(f'  Rango: {df_automl["tasa_abandono_titulacion"].min():.3f} – {df_automl["tasa_abandono_titulacion"].max():.3f}')
print(f'  Media: {df_automl["tasa_abandono_titulacion"].mean():.3f}')
print(f'\n  Planes nuevos → tasa del plan equivalente:')
for plan, equiv in EQUIVALENCIAS_PLANES.items():
    tasa = obtener_tasa(plan, 'SO')  # rama no importa si hay equivalente
    equiv_str = equiv if equiv else 'media rama'
    print(f'  {plan[:55]:<55} → {tasa:.3f} (via {equiv_str[:30]})')
print(f'\n  Media global (fallback): {media_global:.3f}')

# Construir tabla HTML de tasas AQUI — titulacion aun existe
# En la celda siguiente se elimina como leakage
tasas_html = ''
tasas_serie = (
    df_automl.groupby('titulacion')['tasa_abandono_titulacion']
    .first()
    .sort_values(ascending=False)
)
for idx_t, (titulacion_t, tasa_t) in enumerate(tasas_serie.items()):
    bg_t = ' style="background:#f7fafc;"' if idx_t % 2 else ''
    if tasa_t > 0.4:
        color_t = '#e53e3e'
    elif tasa_t > 0.2:
        color_t = '#ed8936'
    else:
        color_t = '#38a169'
    tasas_html += f'<tr{bg_t}><td style="padding:6px;">{titulacion_t}</td>'
    tasas_html += f'<td style="text-align:center;font-weight:bold;color:{color_t};">{tasa_t:.3f}</td></tr>'
print(f'\n  Tabla HTML construida: {len(tasas_serie)} titulaciones')



----------------------------------------------------------------------
TARGET ENCODING: titulacion → tasa_abandono_titulacion
----------------------------------------------------------------------

✓ tasa_abandono_titulacion calculada
  Rango: 0.000 – 0.598
  Media: 0.301

  Planes nuevos → tasa del plan equivalente:
  Grado en Maestro en Educación Primaria (Plan 2018)      → 0.212 (via Grado en Maestro en Educación )
  Grado en Maestro en Educación Infantil (Plan 2018)      → 0.147 (via Grado en Maestro en Educación )
  Grado en Medicina (Plan 2017)                           → 0.120 (via Grado en Medicina)
  Grado en Historia y Patrimonio (Plan 2015)              → 0.469 (via Grado en Historia y Patrimonio)
  Grado en Arquitectura Técnica (Plan 2020)               → 0.382 (via Grado en Arquitectura Técnica)
  Grado en Criminologia y Seguridad  (Plan 2020)          → 0.241 (via Grado en Criminologia y Seguri)
  Grado en Ingeniería Agroalimentaria y del Medio Rural ( → 0.512 (via Grado


✓ tasa_abandono_titulacion calculada
  Rango: 0.000 – 0.598
  Media: 0.301

  Planes nuevos → tasa del plan equivalente:
  Grado en Maestro en Educación Primaria (Plan 2018)      → 0.212 (via Grado en Maestro en Educación )
  Grado en Maestro en Educación Infantil (Plan 2018)      → 0.147 (via Grado en Maestro en Educación )
  Grado en Medicina (Plan 2017)                           → 0.120 (via Grado en Medicina)
  Grado en Historia y Patrimonio (Plan 2015)              → 0.469 (via Grado en Historia y Patrimonio)
  Grado en Arquitectura Técnica (Plan 2020)               → 0.382 (via Grado en Arquitectura Técnica)
  Grado en Criminologia y Seguridad  (Plan 2020)          → 0.241 (via Grado en Criminologia y Seguri)
  Grado en Ingeniería Agroalimentaria y del Medio Rural ( → 0.512 (via Grado en Ingeniería Agroalimen)
  Doble Grado en Administración y Dirección de Empresas y → 0.292 (via media rama)

  Media global (fallback): 0.292

  Tabla HTML construida: 40 titulaciones


In [5]:
# ============================================================================
# CELDA 3b: ELIMINAR COLUMNAS LEAKAGE
# ============================================================================
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  DATA LEAKAGE                                                    ║
# ║  Antes de eliminar: todos los modelos daban AUC=1.0000           ║
# ║  Un DecisionTree depth=3 resolvía:                               ║
# ║    Si curso_ultimo <= 2019 → abandono = 1                       ║
# ║  Esto es la fórmula del target disfrazada.                       ║
# ║                                                                   ║
# ║  Columnas eliminadas:                                             ║
# ║  1. egresado, egresado_de_hecho → en fórmula target              ║
# ║  2. curso_ultimo, anos_inactivo → IS the target                   ║
# ║  3. pct_titulacion, cred_faltantes → implican egresado            ║
# ║  4. per_id_ficticio, exp_tit_id → IDs, no features               ║
# ╚═══════════════════════════════════════════════════════════════════╝

print('\n' + '-'*70)
print('ELIMINANDO COLUMNAS LEAKAGE')
print('-'*70)

# Columnas con leakage directo — contienen la respuesta o la implican
COLS_LEAKAGE = [
    'egresado',            # ES el target disfrazado
    'egresado_de_hecho',   # derivado de egresado
    'curso_ultimo',        # fórmula del target: 2021 - curso_ultimo >= 2
    'anos_inactivo',       # IS the target directamente
    'pct_titulacion',      # implica saber cuánto completó → resultado final
    'cred_faltantes',      # implica saber cuánto le quedaba → resultado final
    'progreso_relativo',   # derivado de pct_titulacion → mismo leakage
    'media_global',        # media de TODOS los años incluyendo el último
    'indicador_casi_termino',  # todos False (campo muerto) + implica resultado
    'titulacion',          # se sustituye por tasa_abandono_titulacion (ver celda anterior)
]
COLS_ID = ['per_id_ficticio', 'exp_tit_id']
COLS_DROP = COLS_LEAKAGE + COLS_ID

resumen_leakage = []
for df_name, df in [('AutoML', df_automl), ('EDA', df_eda)]:
    cols_antes = len(df.columns)
    cols_eliminar = [c for c in COLS_DROP if c in df.columns]
    df.drop(columns=cols_eliminar, inplace=True)
    cols_despues = len(df.columns)
    n_ab = (df['abandono'] == 1).sum()
    resumen_leakage.append({'dataset': df_name, 'antes': cols_antes, 'despues': cols_despues, 'eliminadas': len(cols_eliminar)})
    print(f'\n✓ {df_name}: {cols_antes} → {cols_despues} columnas (-{len(cols_eliminar)})')
    print(f'  Eliminadas: {cols_eliminar}')
    print(f'  Target abandono: {n_ab} ({n_ab/len(df)*100:.1f}%)')
    print(f'  ✅ Sin leakage' if not [c for c in COLS_LEAKAGE if c in df.columns] else f'  ⚠️ LEAKAGE RESTANTE')

print(f'\n✅ Leakage eliminado.')


----------------------------------------------------------------------
ELIMINANDO COLUMNAS LEAKAGE
----------------------------------------------------------------------

✓ AutoML: 52 → 41 columnas (-11)
  Eliminadas: ['egresado', 'egresado_de_hecho', 'curso_ultimo', 'anos_inactivo', 'pct_titulacion', 'cred_faltantes', 'progreso_relativo', 'media_global', 'titulacion', 'per_id_ficticio', 'exp_tit_id']
  Target abandono: 9833 (29.2%)
  ✅ Sin leakage

✓ EDA: 52 → 41 columnas (-11)
  Eliminadas: ['egresado', 'egresado_de_hecho', 'curso_ultimo', 'anos_inactivo', 'pct_titulacion', 'cred_faltantes', 'progreso_relativo', 'media_global', 'titulacion', 'per_id_ficticio', 'exp_tit_id']
  Target abandono: 9833 (29.2%)
  ✅ Sin leakage

✅ Leakage eliminado.


In [6]:
# ============================================================================
# CELDA HTML: DOCUMENTACIÓN M05
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase3', modulo_activo='m05')

n_abandono = (df_automl['abandono'] == 1).sum()
n_total = len(df_automl)

KPIS = [
    {'valor': fmt(n_total), 'titulo': 'Expedientes'},
    {'valor': f'{n_abandono/n_total*100:.1f}%', 'titulo': 'Tasa abandono'},
    {'valor': str(len(COLS_LEAKAGE)), 'titulo': 'Cols leakage eliminadas'},
    {'valor': '2', 'titulo': 'Datasets (D + D_strict)'},
]

s_target = generar_seccion_html('Cálculo del Target', f'''
<div style="background:#f7fafc;padding:20px;border-radius:10px;font-family:monospace;font-size:14px;">
  <strong>abandono</strong> = (egresado = N) AND (egresado_de_hecho = 0) AND (2021 - curso_ultimo &ge; 2)
</div>
<div style="margin-top:15px;display:grid;grid-template-columns:repeat(3,1fr);gap:15px;">
  <div style="background:#f0fff4;padding:12px;border-radius:8px;text-align:center;">
    <div style="font-size:20px;font-weight:bold;color:#38a169;">{(df_automl["abandono"]==0).sum():,}</div>
    <div style="font-size:12px;">No abandona</div>
  </div>
  <div style="background:#fff5f5;padding:12px;border-radius:8px;text-align:center;">
    <div style="font-size:20px;font-weight:bold;color:#e53e3e;">{n_abandono:,}</div>
    <div style="font-size:12px;">Abandona ({n_abandono/n_total*100:.1f}%)</div>
  </div>
  <div style="background:#ebf8ff;padding:12px;border-radius:8px;text-align:center;">
    <div style="font-size:20px;font-weight:bold;color:#3182ce;">{n_total:,}</div>
    <div style="font-size:12px;">Total expedientes</div>
  </div>
</div>''', '🎯')

# Tabla target encoding — construida fuera de f-string para evitar backslash
# tasas_html calculado en celda anterior (target encoding)
# titulacion ya eliminada como leakage — no disponible aqui

s_enc = generar_seccion_html('Target Encoding — titulacion', f'''
<div style="background:#f7fafc;padding:15px;border-radius:10px;margin-bottom:15px;">
  <p><strong>Tecnica:</strong> Target encoding — cada titulacion se sustituye por su tasa historica de abandono.</p>
  <p><strong>Por que:</strong> El numero tiene significado causal real. Titulaciones con alta tasa predicen mayor riesgo.</p>
  <p><strong>Planes nuevos:</strong> Usan la tasa del plan equivalente anterior. Sin equivalente: media de rama.</p>
</div>
<table style="width:100%;border-collapse:collapse;font-size:13px;">
<tr style="background:#3182ce;color:white;">
  <th style="padding:8px;">Titulacion</th>
  <th style="text-align:center;">Tasa abandono</th>
</tr>
{tasas_html}
</table>''', '📊')

s_leakage_html = generar_seccion_html('Data Leakage Eliminado', '''
<div style="background:#fff5f5;padding:15px;border-radius:10px;border-left:4px solid #e53e3e;margin-bottom:15px;">
  <strong>Antes de eliminar:</strong> Todos los modelos AUC=1.0000.<br>
  Un DecisionTree depth=3: Si curso_ultimo &le; 2019 abandono=1.<br>
  El modelo no aprendia — leia la respuesta directamente.
</div>
<div style="background:#f0fff4;padding:15px;border-radius:10px;border-left:4px solid #38a169;">
  <strong>Despues:</strong> AUC=0.93 — resultado real y honesto.
  El modelo aprende patrones genuinos de abandono.
</div>''', '⚠️')

s_datasets = generar_seccion_html('D vs D_strict — Decision de Diseno', '''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#3182ce;color:white;">
  <th style="padding:10px;">Aspecto</th>
  <th style="text-align:center;">D</th>
  <th style="text-align:center;">D_strict</th>
</tr>
<tr><td style="padding:8px;">Features</td><td style="text-align:center;">Mas variables</td><td style="text-align:center;">Solo variables auditadas</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;">Criterio</td><td style="text-align:center;">Sin leakage directo</td><td style="text-align:center;">Sin leakage + sin cuestionables</td></tr>
<tr><td style="padding:8px;">F1</td><td style="text-align:center;">Marginalmente superior</td><td style="text-align:center;">-1.7pp vs D</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;">Produccion</td><td style="text-align:center;">—</td><td style="text-align:center;">Dataset elegido</td></tr>
<tr><td style="padding:8px;">Por que D_strict</td>
  <td colspan="2" style="text-align:center;">Defensa mas solida ante el tribunal. La diferencia de F1 no justifica el riesgo de variables cuestionables.</td>
</tr>
</table>''', '📦')

html = render_pagina_desde_fichero(
    'f3_m05_target_export.ipynb',
    generar_kpis_html(KPIS) + s_target + s_enc + s_leakage_html + s_datasets,
    carpeta_notebook='fase3_features'
)
ruta_html = RUTA_FASE3_HTML / 'm05_target_export.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m05_target_export.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m05_target_export.html


In [7]:
# ============================================================================
# CELDA EXPORTAR: D y D_strict a data/automl/
# ============================================================================
# D        — todas las features sin leakage. Referencia comparativa.
# D_strict — subset auditado. Dataset de producción elegido.
#            Diferencia vs D: −1.7pp F1, pero defensa más sólida.
#
# Features D_strict (confirmadas tras auditoría completa):
# cred_superados_anio_1er, nota_1er_anio, nota_acceso, rama, sexo,
# edad_entrada, pais_nombre, provincia, via_acceso, orden_preferencia,
# universidad_origen, n_anios_beca, situacion_laboral, nota_selectividad,
# max_pagos, indicador_interrupcion, anios_gap, anios_sin_beca,
# tasa_abandono_titulacion, n_anios_trabajando, cred_repetidos,
# tasa_repeticion, n_anios_sin_notas, cupo
# + target: abandono
# ============================================================================

print('\n' + '='*70)
print('EXPORTANDO D y D_strict A data/automl/')
print('='*70)

# Features D_strict — auditadas, sin leakage, sin variables cuestionables
FEATURES_D_STRICT = [
    # Primer año — señal predictiva más fuerte
    'cred_superados_anio_1er', 'nota_1er_anio',
    # Acceso
    'nota_acceso', 'nota_selectividad', 'via_acceso', 'orden_preferencia', 'cupo',
    # Titulación (target encoding)
    'tasa_abandono_titulacion',
    # Académico agregado
    'rama', 'cred_repetidos', 'tasa_repeticion',
    # Demográfico
    'sexo', 'edad_entrada', 'pais_nombre', 'provincia', 'universidad_origen',
    # Beca y laboral
    'n_anios_beca', 'anios_sin_beca', 'situacion_laboral', 'n_anios_trabajando',
    # Económico
    'max_pagos',
    # Indicadores de trayectoria
    'indicador_interrupcion', 'anios_gap', 'n_anios_sin_notas',
    # Target
    'abandono',
]

# Verificar que todas las features existen
features_ok = [f for f in FEATURES_D_STRICT if f in df_automl.columns]
features_missing = [f for f in FEATURES_D_STRICT if f not in df_automl.columns]
if features_missing:
    print(f'⚠️  Features no encontradas: {features_missing}')

# D_strict
df_strict = df_automl[features_ok].copy()
df_strict.to_parquet(RUTA_AUTOML / 'dataset_final_tfm.parquet', index=False)
# Copia en data/03_features/ — ubicación lógica correcta
# data/automl/ se mantiene por compatibilidad con Fase 5, 6 y app
df_strict.to_parquet(RUTA_FEATURES / 'dataset_final_tfm.parquet', index=False)
print(f'\n✓ D_strict: {df_strict.shape[0]:,} × {df_strict.shape[1]} cols')
print(f'  → data/automl/dataset_final_tfm.parquet')

# D — todas las features (sin leakage, sin IDs)
df_D = df_automl.copy()
df_D.to_parquet(RUTA_AUTOML / 'df_exp_automl_D.parquet', index=False)
print(f'\n✓ D: {df_D.shape[0]:,} × {df_D.shape[1]} cols')
print(f'  → data/automl/df_exp_automl_D.parquet')

print(f'\n📊 Diferencia D vs D_strict: {df_D.shape[1] - df_strict.shape[1]} features adicionales en D')
cols_solo_D = [c for c in df_D.columns if c not in df_strict.columns]
print(f'   Features solo en D: {cols_solo_D}')



EXPORTANDO D y D_strict A data/automl/

✓ D_strict: 33,621 × 25 cols
  → data/automl/dataset_final_tfm.parquet

✓ D: 33,621 × 41 cols
  → data/automl/df_exp_automl_D.parquet

📊 Diferencia D vs D_strict: 16 features adicionales en D
   Features solo en D: ['curso_inicio', 'n_cursos', 'cred_matriculados_total', 'cred_superados_total', 'cred_titulacion', 'cred_superados_anio_medio', 'tasa_rendimiento', 'nota_ultimo_anio', 'indicador_edad_inusual', 'indicador_sin_notas', 'duracion_real', 'tiene_gaps', 'ratio_avance', 'velocidad_creditos', 'mejora_notas', 'estabilidad_academica']


In [8]:
# ============================================================================
# CELDA 4b: EXPORTAR DATASET EDA CON TARGET A 03_features/
# ============================================================================
# df_eda tiene: target (celda 3) + leakage eliminado (celda 3b)
# Texto legible. Lo lee M08 para generar df_eda_final.
# TRAZABILIDAD: M04b > M05 (target+leakage) > 03_features/
# ============================================================================

ruta_eda_target = RUTA_FEATURES / "df_eda_con_target.parquet"
df_eda.to_parquet(ruta_eda_target, index=False)

print("\n" + "-"*70)
print("EXPORTAR DATASET EDA CON TARGET")
print("-"*70)
print(f"  Fichero:  {ruta_eda_target.name}")
print(f"  Destino:  data/03_features/")
print(f"  Forma:    {df_eda.shape[0]:,} x {df_eda.shape[1]} cols")
print(f"  Target:   abandono")
print(f"  Texto:    SI (categoricas legibles)")
print(f"  Consumidor: f3_m08_auditoria.ipynb")



----------------------------------------------------------------------
EXPORTAR DATASET EDA CON TARGET
----------------------------------------------------------------------
  Fichero:  df_eda_con_target.parquet
  Destino:  data/03_features/
  Forma:    33,621 x 41 cols
  Target:   abandono
  Texto:    SI (categoricas legibles)
  Consumidor: f3_m08_auditoria.ipynb



EXPORTAR DATASET EDA CON TARGET
----------------------------------------------------------------------
  Fichero:  df_eda_con_target.parquet
  Destino:  data/03_features/
  Forma:    33,621 x 41 cols
  Target:   abandono
  Texto:    SI (categoricas legibles)
  Consumidor: f3_m08_auditoria.ipynb


In [9]:
# ============================================================================
# CELDA 5: RESUMEN FINAL
# ============================================================================

print("\n" + "="*70)
print("✅ F3-M05 COMPLETADO")
print("="*70)
print(f"📊 Registros: {len(df_automl):,}")

print(f"\n📁 ARCHIVOS GENERADOS:")
print(f"\n   📂 data/automl/")
print(f"   ├─ dataset_final_tfm.parquet  ({df_strict.shape[0]:,} x {df_strict.shape[1]} cols) — D_strict: PRODUCCION")
print(f"   └─ df_exp_automl_D.parquet    ({df_D.shape[0]:,} x {df_D.shape[1]} cols) — D: referencia historica")
print(f"\n   📂 data/03_features/")
print(f"   └─ df_eda_con_target.parquet  ({df_eda.shape[0]:,} x {df_eda.shape[1]} cols) — para EDA/CatBoost")

print(f"\n🎯 Dataset de produccion: dataset_final_tfm.parquet")
print(f"   Features: {df_strict.shape[1]-1} variables + target abandono")
if features_missing:
    print(f"\n⚠️  Features no encontradas en df_automl: {features_missing}")
    print("   Revisa que M03 y M04a generaron todas las features correctamente.")
else:
    print("\n✅ Todas las features de D_strict presentes y exportadas correctamente.")

print(f"\n➡️  Siguiente: f3_m06_alerta_temprana.ipynb")
print("="*70)



✅ F3-M05 COMPLETADO
📊 Registros: 33,621

📁 ARCHIVOS GENERADOS:

   📂 data/automl/
   ├─ dataset_final_tfm.parquet  (33,621 x 25 cols) — D_strict: PRODUCCION
   └─ df_exp_automl_D.parquet    (33,621 x 41 cols) — D: referencia historica

   📂 data/03_features/
   └─ df_eda_con_target.parquet  (33,621 x 41 cols) — para EDA/CatBoost

🎯 Dataset de produccion: dataset_final_tfm.parquet
   Features: 24 variables + target abandono

✅ Todas las features de D_strict presentes y exportadas correctamente.

➡️  Siguiente: f3_m06_alerta_temprana.ipynb
